# STAGE 2 — MRI Dataset Pre-processing & Quality Evaluation
### Yugma TechFest 2.0 – MedhaDrishti National-Level AI Hackathon
**Topic:** AI for Medical Image Enhancement and Segmentation

This notebook implements **Stage 2** of the hackathon workflow:

1. Load raw MRI volumes (NIfTI `.nii` / `.nii.gz`) — Brain (T1, T2, FLAIR) and Spine (T1, T2, STIR).
2. Pre-process: denoising, bias-field/artifact correction, rescaling, defocusing non-significant regions.
3. Basic enhancement: Histogram Equalization (HE), Adaptive HE (AHE), Contrast Limited AHE (CLAHE).
4. Image property assessment (Contrast, complexity, Sharpness, Edge strength, Noise level, Mean, Std-Dev) — **before & after**.
5. Full Image Quality Assessment (IQA) suite comparing **raw vs enhanced** slices:
   - PSNR, SSIM, MSE, RMSE, UQI, FSIM, GMSD, VIF (full-reference)
   - BRISQUE, NIQE, PIQE (no-reference / blind)
   - Entropy, LPIPS (learned perceptual)
6. Aggregate results into a CSV report + summary plots.

> **Note:** Update `DATA_DIR` below to point to your local Brain/Spine dataset folders containing the `.nii`/`.nii.gz` files.


In [ ]:
# ================= 0. SETUP & INSTALLS =================
# Run this cell once. Requires internet access to PyPI.
!pip install -q nibabel numpy scipy scikit-image opencv-python-headless \
    matplotlib pandas image-quality piq lpips torch torchvision --no-warn-script-location


In [ ]:
import os, glob, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import nibabel as nib
import cv2
import matplotlib.pyplot as plt

from skimage import exposure, filters, restoration
from skimage.metrics import structural_similarity as sk_ssim
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from scipy.ndimage import uniform_filter, gaussian_filter, sobel
from scipy.stats import entropy as scipy_entropy

import torch

print("Libraries loaded. Torch CUDA available:", torch.cuda.is_available())


## 1. Configuration

Set the dataset root. Expected structure (per hackathon spec):

```
DATA_DIR/
  Brain_MRI/
    Normal/  patient_01/ {T1.nii.gz, T1c.nii.gz, T2.nii.gz, FLAIR.nii.gz}
    Pathological/ patient_01/ {...}
  Spine_MRI/
    Normal/ patient_01/ {T1.nii.gz, T1c.nii.gz, T2.nii.gz, STIR.nii.gz}
    Pathological/ patient_01/ {...}
```

Adjust `DATA_DIR` and the glob pattern in `find_nifti_files()` to match your actual folder layout.


In [ ]:
DATA_DIR = "/path/to/Hackathon_Dataset"   # <-- CHANGE THIS

MODALITIES = ["T1", "T1c", "T2", "FLAIR", "STIR"]

def find_nifti_files(root):
    """Recursively find all .nii / .nii.gz files under root."""
    patterns = ["**/*.nii", "**/*.nii.gz"]
    files = []
    for p in patterns:
        files.extend(glob.glob(os.path.join(root, p), recursive=True))
    return sorted(files)

nifti_files = find_nifti_files(DATA_DIR)
print(f"Found {len(nifti_files)} NIfTI files.")
for f in nifti_files[:10]:
    print(" ", f)


## 2. Loading MRI Volumes & Extracting Representative Slices

For 3D volume processing we work slice-by-slice on the axial (or sagittal, for spine) plane.
Each slice is normalized to `[0, 255]` uint8 for classical 2D enhancement algorithms,
while the original float volume is preserved for any 3D-aware steps.


In [ ]:
def load_nifti(path):
    img = nib.load(path)
    vol = img.get_fdata().astype(np.float32)
    return vol, img.affine, img.header

def normalize_to_uint8(slice_2d):
    """Min-max normalize a 2D slice to 0-255 uint8, robust to constant slices."""
    s = slice_2d.astype(np.float32)
    lo, hi = np.percentile(s, 0.5), np.percentile(s, 99.5)
    if hi - lo < 1e-6:
        return np.zeros_like(s, dtype=np.uint8)
    s = np.clip((s - lo) / (hi - lo), 0, 1)
    return (s * 255).astype(np.uint8)

def get_middle_slice(vol, axis=2):
    """Extract the middle slice along a given axis (2 = axial for typical NIfTI orientation)."""
    idx = vol.shape[axis] // 2
    if axis == 0:
        return vol[idx, :, :]
    elif axis == 1:
        return vol[:, idx, :]
    else:
        return vol[:, :, idx]

def get_informative_slice(vol, axis=2, n_candidates=5):
    """Pick the slice with the highest non-zero pixel fraction near the volume center
    (avoids picking an all-background slice for pathological/edge cases)."""
    depth = vol.shape[axis]
    center = depth // 2
    span = max(1, depth // 6)
    candidates = range(max(0, center - span), min(depth, center + span))
    best_idx, best_score = center, -1
    for i in candidates:
        sl = np.take(vol, i, axis=axis)
        score = np.count_nonzero(sl > np.percentile(vol, 10))
        if score > best_score:
            best_score, best_idx = score, i
    return np.take(vol, best_idx, axis=axis), best_idx


## 3. Pre-processing Pipeline

Steps applied to each 2D slice (as per Stage 2 requirements):

1. **Denoising** — Non-Local Means (NLM) denoising, edge-preserving.
2. **Bias-field / intensity inhomogeneity correction** — simplified homomorphic filtering
   (a full N4ITK bias correction can be substituted using SimpleITK if available).
3. **Artifact correction** — median filtering for salt-and-pepper / spike artifacts.
4. **Background defocusing** — suppress non-significant (background/air) regions using an Otsu mask.
5. **Rescaling** — resize to a common resolution (e.g., 256x256) for uniform downstream processing.


In [ ]:
def denoise_nlm(img_uint8, h=8, template=7, search=21):
    return cv2.fastNlMeansDenoising(img_uint8, None, h, template, search)

def median_artifact_correction(img_uint8, ksize=3):
    return cv2.medianBlur(img_uint8, ksize)

def homomorphic_bias_correction(img_uint8, sigma=30):
    """Simplified homomorphic filtering to reduce low-frequency intensity inhomogeneity
    (bias field) — log domain high-pass filtering."""
    img_f = img_uint8.astype(np.float32) + 1.0
    log_img = np.log(img_f)
    low_freq = gaussian_filter(log_img, sigma=sigma)
    high_freq = log_img - low_freq
    corrected = np.exp(high_freq + np.mean(low_freq))
    corrected = normalize_to_uint8(corrected)
    return corrected

def defocus_background(img_uint8):
    """Otsu-threshold based background suppression — keeps anatomical ROI, dims background/air."""
    _, mask = cv2.threshold(img_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
    background_dimmed = img_uint8.copy()
    background_dimmed[mask == 0] = (img_uint8[mask == 0] * 0.3).astype(np.uint8)
    return background_dimmed, mask

def rescale_slice(img_uint8, size=(256, 256)):
    return cv2.resize(img_uint8, size, interpolation=cv2.INTER_CUBIC)

def preprocess_pipeline(raw_slice, target_size=(256, 256)):
    """Full Stage-2 preprocessing pipeline. Returns dict of intermediate + final outputs."""
    img0 = normalize_to_uint8(raw_slice)
    img1 = rescale_slice(img0, target_size)
    img2 = median_artifact_correction(img1, ksize=3)
    img3 = denoise_nlm(img2)
    img4 = homomorphic_bias_correction(img3)
    img5, mask = defocus_background(img4)
    return {
        "raw": img1,
        "artifact_corrected": img2,
        "denoised": img3,
        "bias_corrected": img4,
        "preprocessed": img5,
        "roi_mask": mask,
    }


## 4. Basic MRI Enhancement Methods

- **HE** — Global Histogram Equalization
- **AHE** — Adaptive Histogram Equalization (block-wise, no clip limit)
- **CLAHE** — Contrast Limited Adaptive Histogram Equalization


In [ ]:
def apply_he(img_uint8):
    return cv2.equalizeHist(img_uint8)

def apply_ahe(img_uint8, clip_limit_disabled=True):
    # AHE = CLAHE with a very high clip limit (effectively unclipped)
    clahe = cv2.createCLAHE(clipLimit=100.0, tileGridSize=(8, 8))
    return clahe.apply(img_uint8)

def apply_clahe(img_uint8, clip_limit=2.0, tile_grid=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(img_uint8)

def enhance_all(preprocessed_img):
    return {
        "HE": apply_he(preprocessed_img),
        "AHE": apply_ahe(preprocessed_img),
        "CLAHE": apply_clahe(preprocessed_img),
    }


## 5. Image Property Assessment
(Contrast, Complexity, Sharpness, Edge Strength, Noise Level, Mean, Std-Dev)

Applied identically to raw, pre-processed, and enhanced images so results are directly comparable
across Stage 1 → Stage 2 → Stage 3.


In [ ]:
def image_properties(img_uint8):
    img = img_uint8.astype(np.float64)

    mean_val = np.mean(img)
    std_val = np.std(img)

    # Contrast: Michelson contrast style (using percentiles to avoid outlier sensitivity)
    i_max, i_min = np.percentile(img, 99), np.percentile(img, 1)
    contrast = (i_max - i_min) / (i_max + i_min + 1e-8)

    # Complexity: normalized Shannon entropy of intensity histogram
    hist, _ = np.histogram(img, bins=256, range=(0, 255), density=True)
    complexity = scipy_entropy(hist + 1e-12, base=2)

    # Sharpness: variance of Laplacian
    lap = cv2.Laplacian(img_uint8, cv2.CV_64F)
    sharpness = lap.var()

    # Edge strength: mean gradient magnitude (Sobel)
    gx = sobel(img, axis=0)
    gy = sobel(img, axis=1)
    edge_strength = np.mean(np.sqrt(gx**2 + gy**2))

    # Noise level: estimated via high-frequency residual (img - blurred img) std, in a flat region proxy
    blurred = gaussian_filter(img, sigma=1.5)
    noise_level = np.std(img - blurred)

    return {
        "Mean": mean_val,
        "StdDev": std_val,
        "Contrast": contrast,
        "Complexity(Entropy)": complexity,
        "Sharpness(LapVar)": sharpness,
        "EdgeStrength": edge_strength,
        "NoiseLevel": noise_level,
    }


## 6. Image Quality Evaluation Metrics

### 6.1 Full-Reference Metrics (require raw vs enhanced pair)
PSNR, SSIM, MSE, RMSE, UQI, FSIM, GMSD, VIF


In [ ]:
def calc_mse(ref, test):
    ref = ref.astype(np.float64); test = test.astype(np.float64)
    return np.mean((ref - test) ** 2)

def calc_rmse(ref, test):
    return np.sqrt(calc_mse(ref, test))

def calc_psnr(ref, test, data_range=255):
    mse = calc_mse(ref, test)
    if mse == 0:
        return float('inf')
    return sk_psnr(ref, test, data_range=data_range)

def calc_ssim(ref, test, data_range=255):
    return sk_ssim(ref, test, data_range=data_range)

def calc_uqi(ref, test, block_size=8):
    """Universal Image Quality Index (Wang & Bovik, 2002), computed on sliding blocks."""
    ref = ref.astype(np.float64); test = test.astype(np.float64)
    N = block_size * block_size
    kernel = np.ones((block_size, block_size)) / N

    mu_x = cv2.filter2D(ref, -1, kernel)
    mu_y = cv2.filter2D(test, -1, kernel)
    mu_x2 = cv2.filter2D(ref**2, -1, kernel)
    mu_y2 = cv2.filter2D(test**2, -1, kernel)
    mu_xy = cv2.filter2D(ref*test, -1, kernel)

    var_x = mu_x2 - mu_x**2
    var_y = mu_y2 - mu_y**2
    cov_xy = mu_xy - mu_x*mu_y

    numerator = 4 * cov_xy * mu_x * mu_y
    denominator = (var_x + var_y) * (mu_x**2 + mu_y**2)
    denominator = np.where(denominator == 0, 1e-8, denominator)
    q_map = numerator / denominator
    return np.mean(q_map)

def _gradient_magnitude(img):
    gx = sobel(img.astype(np.float64), axis=0)
    gy = sobel(img.astype(np.float64), axis=1)
    return np.sqrt(gx**2 + gy**2)

def calc_fsim(ref, test):
    """Simplified Feature Similarity Index using phase congruency proxy (gradient-based)
    and gradient magnitude similarity, following the FSIM formulation (Zhang et al., 2011)."""
    ref = ref.astype(np.float64); test = test.astype(np.float64)

    g_ref = _gradient_magnitude(ref)
    g_test = _gradient_magnitude(test)

    T1, T2 = 0.85, 160
    pc_ref = g_ref / (g_ref.max() + 1e-8)   # phase-congruency proxy
    pc_test = g_test / (g_test.max() + 1e-8)

    S_pc = (2 * pc_ref * pc_test + T1) / (pc_ref**2 + pc_test**2 + T1)
    S_g = (2 * g_ref * g_test + T2) / (g_ref**2 + g_test**2 + T2)
    S_l = S_pc * S_g

    pc_max = np.maximum(pc_ref, pc_test)
    fsim = np.sum(S_l * pc_max) / (np.sum(pc_max) + 1e-8)
    return fsim

def calc_gmsd(ref, test):
    """Gradient Magnitude Similarity Deviation (Xue et al., 2014). Lower = more similar."""
    g_ref = _gradient_magnitude(ref)
    g_test = _gradient_magnitude(test)
    c = 170.0
    gms = (2 * g_ref * g_test + c) / (g_ref**2 + g_test**2 + c)
    gmsd = np.std(gms)
    return gmsd

def calc_vif(ref, test, sigma_nsq=2.0):
    """Visual Information Fidelity (simplified, single-scale, Sheikh & Bovik, 2006),
    computed via local Gaussian-window statistics."""
    ref = ref.astype(np.float64); test = test.astype(np.float64)
    eps = 1e-10
    num, den = 0.0, 0.0
    scales = [2, 4, 8]
    for scale in scales:
        sigma = scale / 5.0
        mu_ref = gaussian_filter(ref, sigma)
        mu_test = gaussian_filter(test, sigma)
        var_ref = gaussian_filter(ref**2, sigma) - mu_ref**2
        var_test = gaussian_filter(test**2, sigma) - mu_test**2
        cov = gaussian_filter(ref*test, sigma) - mu_ref*mu_test

        var_ref = np.clip(var_ref, eps, None)
        g = cov / var_ref
        sigma_v_sq = var_test - g * cov
        sigma_v_sq = np.clip(sigma_v_sq, eps, None)

        num += np.sum(np.log10(1 + (g**2 * var_ref) / (sigma_v_sq + sigma_nsq)))
        den += np.sum(np.log10(1 + var_ref / sigma_nsq))
    return num / (den + eps)


### 6.2 No-Reference (Blind) Metrics
BRISQUE, NIQE, PIQE, Entropy — assess quality without needing a "clean" reference,
useful since raw clinical MRI has no ground-truth clean version.


In [ ]:
# ---- BRISQUE (via the 'image-quality' / piq packages if available, else fallback) ----
def calc_brisque(img_uint8):
    try:
        import piq
        import torch
        t = torch.tensor(img_uint8, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0
        t3 = t.repeat(1, 3, 1, 1)  # BRISQUE expects RGB-like input
        score = piq.brisque(t3, data_range=1.0, reduction='mean')
        return float(score.item())
    except Exception as e:
        # Fallback: simple NSS-based proxy using local contrast normalization statistics
        img = img_uint8.astype(np.float64)
        mu = gaussian_filter(img, 7/6)
        mu_sq = mu * mu
        sigma = np.sqrt(np.abs(gaussian_filter(img*img, 7/6) - mu_sq))
        mscn = (img - mu) / (sigma + 1)
        return float(np.var(mscn))  # heuristic proxy, lower = more natural

def calc_entropy(img_uint8):
    hist, _ = np.histogram(img_uint8, bins=256, range=(0, 255), density=True)
    return scipy_entropy(hist + 1e-12, base=2)

def calc_niqe(img_uint8):
    """NIQE proxy using local MSCN (Mean Subtracted Contrast Normalized) coefficient statistics
    against a naturalness assumption (Mittal et al., 2013). Uses 'piq' package if available."""
    try:
        import piq, torch
        t = torch.tensor(img_uint8, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0
        score = piq.niqe(t, data_range=1.0)
        return float(score.item())
    except Exception:
        img = img_uint8.astype(np.float64)
        mu = gaussian_filter(img, 7/6)
        sigma = np.sqrt(np.abs(gaussian_filter(img*img, 7/6) - mu*mu))
        mscn = (img - mu) / (sigma + 1)
        # naturalness deviation proxy: distance of skew/kurtosis from Gaussian(0,1)
        from scipy.stats import skew, kurtosis
        return float(abs(skew(mscn.ravel())) + abs(kurtosis(mscn.ravel())))

def calc_piqe(img_uint8, block_size=16):
    """PIQE proxy (Perception based Image Quality Evaluator, Venkatanath et al., 2015):
    fraction of 'distorted' blocks based on local variance thresholding."""
    img = img_uint8.astype(np.float64)
    h, w = img.shape
    distorted_blocks = 0
    total_blocks = 0
    scores = []
    for i in range(0, h - block_size, block_size):
        for j in range(0, w - block_size, block_size):
            block = img[i:i+block_size, j:j+block_size]
            var = np.var(block)
            total_blocks += 1
            if var < 4.0:  # near-uniform / low-activity block -> flagged
                distorted_blocks += 1
            else:
                scores.append(var)
    activity_penalty = 100.0 * distorted_blocks / max(total_blocks, 1)
    return activity_penalty


### 6.3 Learned Perceptual Metric — LPIPS

Uses a pretrained deep network (AlexNet/VGG backbone) to compute perceptual distance between
raw and enhanced images. Requires the `lpips` package (installed above).


In [ ]:
_lpips_model = None
def get_lpips_model():
    global _lpips_model
    if _lpips_model is None:
        import lpips
        _lpips_model = lpips.LPIPS(net='alex')
    return _lpips_model

def calc_lpips(ref_uint8, test_uint8):
    import torch
    model = get_lpips_model()
    def to_tensor(img):
        img3 = np.stack([img]*3, axis=0).astype(np.float32) / 127.5 - 1.0  # [-1, 1], 3xHxW
        return torch.tensor(img3).unsqueeze(0)
    t_ref = to_tensor(ref_uint8)
    t_test = to_tensor(test_uint8)
    with torch.no_grad():
        d = model(t_ref, t_test)
    return float(d.item())


## 7. Metric Aggregator

Runs the full IQA suite (full-reference + no-reference + LPIPS) comparing a **raw** image
against a **candidate** (preprocessed/enhanced) image.


In [ ]:
def evaluate_all_metrics(raw_img, candidate_img):
    results = {}
    # Full-reference
    results["PSNR"]  = calc_psnr(raw_img, candidate_img)
    results["SSIM"]  = calc_ssim(raw_img, candidate_img)
    results["MSE"]   = calc_mse(raw_img, candidate_img)
    results["RMSE"]  = calc_rmse(raw_img, candidate_img)
    results["UQI"]   = calc_uqi(raw_img, candidate_img)
    results["FSIM"]  = calc_fsim(raw_img, candidate_img)
    results["GMSD"]  = calc_gmsd(raw_img, candidate_img)
    results["VIF"]   = calc_vif(raw_img, candidate_img)
    # No-reference (computed on the candidate image itself)
    results["BRISQUE"] = calc_brisque(candidate_img)
    results["NIQE"]    = calc_niqe(candidate_img)
    results["PIQE"]    = calc_piqe(candidate_img)
    results["Entropy"] = calc_entropy(candidate_img)
    # Learned perceptual
    try:
        results["LPIPS"] = calc_lpips(raw_img, candidate_img)
    except Exception as e:
        results["LPIPS"] = np.nan
        print("LPIPS failed:", e)
    return results


## 8. End-to-End Run: Preprocess + Enhance + Evaluate All NIfTI Files

For each NIfTI file found:
1. Load volume → extract an informative slice.
2. Run preprocessing pipeline.
3. Apply HE / AHE / CLAHE enhancement.
4. Compute image properties for raw, preprocessed, and each enhanced variant.
5. Compute full IQA metric suite (raw vs preprocessed, raw vs HE/AHE/CLAHE).
6. Collect everything into a results table.


In [ ]:
all_rows = []

for fpath in nifti_files:
    fname = os.path.basename(fpath)
    print(f"Processing: {fname}")
    try:
        vol, affine, header = load_nifti(fpath)
        if vol.ndim < 2:
            continue
        if vol.ndim == 2:
            raw_slice = vol
        else:
            raw_slice, slice_idx = get_informative_slice(vol, axis=2)

        pp = preprocess_pipeline(raw_slice)
        raw_img = pp["raw"]
        preprocessed_img = pp["preprocessed"]
        enhanced = enhance_all(preprocessed_img)

        candidates = {"Preprocessed": preprocessed_img, **enhanced}

        for cand_name, cand_img in candidates.items():
            props = image_properties(cand_img)
            metrics = evaluate_all_metrics(raw_img, cand_img)
            row = {"file": fname, "stage": cand_name}
            row.update(props)
            row.update(metrics)
            all_rows.append(row)

        # also log raw image properties once
        raw_props = image_properties(raw_img)
        raw_row = {"file": fname, "stage": "Raw"}
        raw_row.update(raw_props)
        all_rows.append(raw_row)

    except Exception as e:
        print(f"  ERROR processing {fname}: {e}")

results_df = pd.DataFrame(all_rows)
results_df.to_csv("/mnt/user-data/outputs/stage2_iqa_results.csv", index=False)
results_df.head(20)


## 9. Visual Comparison (Sample)

Shows Raw vs Preprocessed vs HE vs AHE vs CLAHE for a single sample file, to visually confirm
enhancement quality alongside the quantitative metrics above.


In [ ]:
if len(nifti_files) > 0:
    sample_path = nifti_files[0]
    vol, _, _ = load_nifti(sample_path)
    raw_slice, _ = get_informative_slice(vol, axis=2) if vol.ndim == 3 else (vol, None)
    pp = preprocess_pipeline(raw_slice)
    enhanced = enhance_all(pp["preprocessed"])

    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    titles = ["Raw", "Preprocessed", "HE", "AHE", "CLAHE"]
    imgs = [pp["raw"], pp["preprocessed"], enhanced["HE"], enhanced["AHE"], enhanced["CLAHE"]]
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(img, cmap="gray")
        ax.set_title(title)
        ax.axis("off")
    plt.suptitle(f"Sample: {os.path.basename(sample_path)}")
    plt.tight_layout()
    plt.savefig("/mnt/user-data/outputs/stage2_sample_comparison.png", dpi=150)
    plt.show()
else:
    print("No NIfTI files found — check DATA_DIR.")


## 10. Summary Statistics per Enhancement Method

Aggregated mean ± std of all metrics, grouped by enhancement stage (Preprocessed / HE / AHE / CLAHE),
to identify which method yields the best trade-off (e.g., highest SSIM/PSNR, lowest BRISQUE/NIQE/GMSD).


In [ ]:
summary = results_df[results_df["stage"] != "Raw"].groupby("stage").agg(
    ["mean", "std"]
)
summary_numeric = summary.select_dtypes(include=[np.number])
summary_numeric.to_csv("/mnt/user-data/outputs/stage2_summary_by_method.csv")
summary_numeric


## 11. Next Steps (Stage 3)

The `Preprocessed` outputs (and their evaluation baselines above) feed directly into **Stage 3**,
where a deep-learning enhancement model (single CNN/Transformer or ensemble) will be trained and
compared against the classical HE/AHE/CLAHE baselines using this same metric suite, plus training
diagnostics (loss curves, convergence epoch, overfitting gap) and efficiency metrics (latency,
throughput, GPU/CPU utilization, memory, model complexity) as required by the hackathon rubric.
